# SOH Forecasting — Multi-vehicle training, held-out Vehicle 14

Same dQ/ΔV SOH pipeline and Transformer-encoder architecture as `soh_forecast_dqdv_without_SOC_cell_voltage.ipynb`, but trained on **five vehicles combined** (1, 3, 8, 10, 15) and tested on the held-out **Vehicle 14**.

## Key design choices

- **Per-vehicle aggregation:** each CSV is segmented on a 1-minute gap, charging-only sessions with ≥ `MIN_ROWS` samples are kept, and the first 1000 segments are retained.
- **Per-vehicle SOH calibration:** each vehicle's `(dQ/ΔV)_ref` comes from **its own** first `REF_HEAD` full-charge segments, so SOH starts near 100% for every vehicle.
- **No `vehicle_id` feature.** The model has to be vehicle-agnostic to generalise to Vehicle 14.
- **Windows do not cross vehicle boundaries.** Built per vehicle, then concatenated for training.
- **Test starts at Vehicle 14 segment `WINDOW` (= 180).** The first 180 segments of Vehicle 14 form the input window for the first prediction, then we forecast every segment 180..N-1.
- **Validation:** 10 % of training windows, shuffled across vehicles.

> Requires: `pandas numpy matplotlib scikit-learn torch`

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

torch.manual_seed(0); np.random.seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Configuration

One entry per vehicle. Each vehicle gets its own SOH reference computed later from its own early-life full-charge segments. Paths are relative to this notebook's working directory.

In [ ]:
VEHICLE_CONFIG = {
    1:  {"path": "#1.csv",  "role": "train"},
    3:  {"path": "#3.csv",  "role": "train"},
    8:  {"path": "#8.csv",  "role": "train"},
    10: {"path": "#10.csv", "role": "train"},
    15: {"path": "#15.csv", "role": "train"},
    14: {"path": "#14.csv", "role": "test"},
}

# Segmentation / aggregation
GAP_MINUTES = 1
MIN_ROWS    = 10
N_KEEP      = 1000

# SOH (dQ/ΔV) derivation knobs
FULL_DELTA_SOC = 60.0
MIN_DELTA_V    = 5.0
REF_HEAD       = 5
ROLL_WINDOW    = 5

# Windowing
WINDOW  = 180
HORIZON = 1

RENAME = {
    "pack_voltage (V)":        "pack_voltage",
    "charge_current (A)":      "charge_current",
    "max_cell_voltage (V)":    "max_cell_voltage",
    "min_cell_voltage (V)":    "min_cell_voltage",
    "max_temperature (℃)":  "max_temperature",
    "min_temperature (℃)":  "min_temperature",
    "available_energy (kw)":   "available_energy",
    "available_capacity (Ah)": "available_capacity",
}

## 2. Per-vehicle load + segment + aggregate

Each CSV is read independently, sessions are split on a 1-minute gap, and only charging-current sessions with at least `MIN_ROWS` samples are kept. The aggregated per-segment table is truncated to the first 1000 segments.

In [ ]:
def aggregate_session(g):
    cur = g["charge_current"]
    if cur.median() >= -1.0:
        return None
    if len(g) < MIN_ROWS:
        return None
    duration_s = (g["timestamp"].iloc[-1] - g["timestamp"].iloc[0]).total_seconds()
    v_start = g["pack_voltage"].iloc[0]; v_end = g["pack_voltage"].iloc[-1]
    soc_start = g["soc"].iloc[0]; soc_end = g["soc"].iloc[-1]
    return pd.Series({
        "t_start":              g["timestamp"].iloc[0],
        "t_end":                g["timestamp"].iloc[-1],
        "duration_min":         duration_s / 60.0,
        "soc_start":            soc_start,
        "soc_end":              soc_end,
        "delta_soc":            soc_end - soc_start,
        "v_pack_mean":          g["pack_voltage"].mean(),
        "v_pack_max":           g["pack_voltage"].max(),
        "v_pack_min":           g["pack_voltage"].min(),
        "v_start":              v_start,
        "v_end":                v_end,
        "delta_v":              v_end - v_start,
        "i_chg_mean_abs":       cur.abs().mean(),
        "i_chg_max_abs":        cur.abs().max(),
        "temp_max_mean":        g["max_temperature"].mean(),
        "temp_min_mean":        g["min_temperature"].mean(),
        "temp_spread_mean":     (g["max_temperature"] - g["min_temperature"]).mean(),
        "cell_v_max_mean":      g["max_cell_voltage"].mean(),
        "cell_v_spread_mean":   (g["max_cell_voltage"] - g["min_cell_voltage"]).mean(),
        "cap_charged_ah":       g["available_capacity"].iloc[-1] - g["available_capacity"].iloc[0],
        "energy_charged_kwh":   g["available_energy"].iloc[-1]   - g["available_energy"].iloc[0],
        "available_capacity_max": g["available_capacity"].max(),
    })

def load_and_aggregate(vehicle_id, path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={df.columns[0]: "row_idx", **RENAME})
    missing = [v for v in RENAME.values() if v not in df.columns]
    assert not missing, f"Vehicle {vehicle_id}: missing columns {missing}. Got: {list(df.columns)}"
    df["timestamp"] = pd.to_datetime(df["record_time"], format="%Y%m%d%H%M%S")
    df = df.sort_values("timestamp").reset_index(drop=True)

    dt = df["timestamp"].diff().dt.total_seconds().fillna(0)
    df["session"] = (dt > GAP_MINUTES * 60).cumsum()

    records = []
    for sid, g in df.groupby("session", sort=True):
        rec = aggregate_session(g)
        if rec is not None:
            rec["session"] = sid
            records.append(rec)

    cycles = pd.DataFrame(records).reset_index(drop=True)
    cycles = cycles.iloc[:N_KEEP].reset_index(drop=True)
    cycles["cycle_idx"] = np.arange(len(cycles))
    cycles["days_since_start"] = (cycles["t_start"] - cycles["t_start"].iloc[0]).dt.total_seconds() / 86400.0
    cycles["vehicle_id"] = vehicle_id
    return cycles

## 3. Per-vehicle SOH derivation (dQ/ΔV formula)

For each vehicle:
1. Pick **full-charge segments** with `delta_soc ≥ FULL_DELTA_SOC` and `delta_v ≥ MIN_DELTA_V`.
2. Compute `dQ/ΔV` on those anchors.
3. Reference = mean of the first `REF_HEAD` anchors.
4. `SOH% = 100 * (dQ/ΔV) / ref`, smoothed with a centred rolling median.
5. Linearly interpolate the smoothed SOH onto every segment so each segment has a label.

In [ ]:
def compute_soh(cycles, ref_head=REF_HEAD, roll_window=ROLL_WINDOW,
                full_delta_soc=FULL_DELTA_SOC, min_delta_v=MIN_DELTA_V):
    full_mask = (cycles["delta_soc"] >= full_delta_soc) & (cycles["delta_v"] >= min_delta_v)
    full = cycles[full_mask].copy().reset_index(drop=True)
    if len(full) < ref_head:
        raise ValueError(f"Only {len(full)} full segments found; need at least {ref_head} for the reference.")

    full["dq_over_dv"]     = full["cap_charged_ah"] / full["delta_v"]
    ref                    = full["dq_over_dv"].iloc[:ref_head].mean()
    full["soh_pct_raw"]    = 100.0 * full["dq_over_dv"] / ref
    full["soh_pct_smooth"] = full["soh_pct_raw"].rolling(roll_window, min_periods=1, center=True).median()

    cycles = cycles.copy()
    cycles["soh_pct"] = np.interp(cycles["cycle_idx"], full["cycle_idx"], full["soh_pct_smooth"])
    return cycles, {"ref_dq_over_dv": float(ref), "n_full_segments": int(len(full))}

## 4. Run aggregation + SOH for all six vehicles

In [ ]:
all_cycles = {}
all_refs   = {}
for vid, cfg in VEHICLE_CONFIG.items():
    cyc = load_and_aggregate(vid, cfg["path"])
    cyc, refs = compute_soh(cyc)
    all_cycles[vid] = cyc
    all_refs[vid]   = refs
    print(f"Vehicle {vid:2d} ({cfg['role']:5s}): {len(cyc):4d} segments  "
          f"full-anchors={refs['n_full_segments']:3d}  "
          f"(dQ/ΔV)_ref={refs['ref_dq_over_dv']:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for vid, cyc in all_cycles.items():
    role = VEHICLE_CONFIG[vid]["role"]
    ls = "-" if role == "train" else "--"
    ax.plot(cyc["cycle_idx"], cyc["soh_pct"], ls, alpha=0.85, label=f"Vehicle {vid} ({role})")
ax.set_xlabel("segment index (within vehicle)"); ax.set_ylabel("SOH %"); ax.legend()
ax.set_title("Per-vehicle smoothed SOH (each anchored to its own early-life reference)")
plt.show()

## 5. Build windows per vehicle, combine training, separate test

Windows do not cross vehicle boundaries. For each vehicle, `make_windows` builds (X, y) pairs where the input is 180 consecutive segments and the target is the SOH of the next segment.

In [ ]:
FEATURE_COLS = [
    "duration_min",
    "v_pack_mean", "v_pack_max", "v_pack_min", "delta_v",
    "i_chg_mean_abs", "i_chg_max_abs",
    "temp_max_mean", "temp_min_mean", "temp_spread_mean",
    "cap_charged_ah", "energy_charged_kwh",
    "available_capacity_max",
    "days_since_start",
    "soh_pct",  # current SOH as input feature for next-segment forecast
]
print("Features:", len(FEATURE_COLS))

def make_windows(cyc, window=WINDOW, horizon=HORIZON):
    X = cyc[FEATURE_COLS].values.astype(np.float32)
    y = cyc["soh_pct"].values.astype(np.float32)
    Xs, ys = [], []
    for i in range(len(cyc) - window - horizon + 1):
        Xs.append(X[i:i+window])
        ys.append(y[i + window + horizon - 1])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

# Training windows from vehicles 1, 3, 8, 10, 15
train_X_blocks, train_y_blocks = [], []
for vid, cfg in VEHICLE_CONFIG.items():
    if cfg["role"] != "train":
        continue
    Xc, yc = make_windows(all_cycles[vid])
    train_X_blocks.append(Xc); train_y_blocks.append(yc)
    print(f"  Vehicle {vid:2d}: {len(Xc)} windows")
X_train_all = np.concatenate(train_X_blocks, axis=0)
y_train_all = np.concatenate(train_y_blocks, axis=0)
print(f"Combined train: X {X_train_all.shape}  y {y_train_all.shape}")

# Test windows from Vehicle 14 (test-cycle indices = WINDOW, WINDOW+1, ..., N-1)
vehicle_test = all_cycles[14]
X_test, y_test = make_windows(vehicle_test)
test_cycle_idx = np.arange(WINDOW + HORIZON - 1, len(vehicle_test))
print(f"Test (Vehicle 14): {len(X_test)} windows, predicting cycle_idx {test_cycle_idx[0]}..{test_cycle_idx[-1]}")

In [ ]:
# StandardScaler fit on training feature rows only, applied to both train and test
scaler = StandardScaler().fit(X_train_all.reshape(-1, X_train_all.shape[-1]))
def scale(X):
    flat = X.reshape(-1, X.shape[-1])
    return scaler.transform(flat).reshape(X.shape).astype(np.float32)

X_train_all = scale(X_train_all)
X_test      = scale(X_test)
print("Scaled.")

In [ ]:
# Shuffle training windows across vehicles, carve a 10% validation slice
rng = np.random.RandomState(0)
idx = rng.permutation(len(X_train_all))
val_n = max(64, int(0.1 * len(X_train_all)))
val_idx   = idx[:val_n]
train_idx = idx[val_n:]
X_tr, y_tr = X_train_all[train_idx], y_train_all[train_idx]
X_vl, y_vl = X_train_all[val_idx],   y_train_all[val_idx]
print(f"train {len(X_tr)}  val {len(X_vl)}  test {len(X_test)}")

class WinDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_dl = DataLoader(WinDS(X_tr, y_tr),     batch_size=64, shuffle=True)
val_dl   = DataLoader(WinDS(X_vl, y_vl),     batch_size=128)
test_dl  = DataLoader(WinDS(X_test, y_test), batch_size=128)

## 6. Transformer encoder model

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class SOHTransformer(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, layers=3, dim_ff=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos = PositionalEncoding(d_model)
        enc = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(enc, num_layers=layers)
        self.head = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, 1))
    def forward(self, x):
        h = self.pos(self.input_proj(x)); h = self.encoder(h)
        return self.head(h[:, -1]).squeeze(-1)

model = SOHTransformer(n_features=len(FEATURE_COLS)).to(DEVICE)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7. Train

In [ ]:
EPOCHS = 80
LR = 1e-3
PATIENCE = 12

opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.MSELoss()

best_val = float("inf"); best_state = None; bad = 0
hist = {"train": [], "val": []}

for epoch in range(EPOCHS):
    model.train(); tr = 0.0; n = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr += loss.item() * len(xb); n += len(xb)
    sched.step(); tr /= n

    model.eval(); vl = 0.0; nv = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            vl += loss_fn(model(xb), yb).item() * len(xb); nv += len(xb)
    vl /= nv
    hist["train"].append(tr); hist["val"].append(vl)

    if vl < best_val - 1e-6:
        best_val = vl
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:3d}  train {tr:.5f}  val {vl:.5f}  best {best_val:.5f}")
    if bad >= PATIENCE:
        print(f"Early stop at epoch {epoch}, best val {best_val:.5f}")
        break

model.load_state_dict(best_state)

plt.figure(figsize=(7, 3))
plt.plot(hist["train"], label="train"); plt.plot(hist["val"], label="val")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend(); plt.title("Loss")
plt.show()

## 8. Evaluate on Vehicle 14

Forecast every segment from cycle_idx `WINDOW` onwards using teacher-forced rolling 180-segment windows. Persistence (last seen SOH) and a linear extrapolation fit on Vehicle 14's warm-up window are included as naive baselines.

In [ ]:
@torch.no_grad()
def predict(model, dl):
    model.eval()
    out = []
    for xb, _ in dl:
        out.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(out)

y_pred = predict(model, test_dl)

# Persistence: last SOH in each test window (unscaled back)
soh_idx = FEATURE_COLS.index("soh_pct")
mu, sg  = scaler.mean_[soh_idx], scaler.scale_[soh_idx]
y_persist = X_test[:, -1, soh_idx] * sg + mu

# Linear extrapolation from Vehicle 14's first WINDOW segments
warmup = vehicle_test["soh_pct"].values[:WINDOW]
coef = np.polyfit(np.arange(WINDOW), warmup, 1)
y_linear = np.polyval(coef, test_cycle_idx)

def report(name, yt, yp):
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    print(f"{name:14s}  MAE {mean_absolute_error(yt, yp):.4f}  RMSE {rmse:.4f}  R2 {r2_score(yt, yp):.4f}")

report("Transformer", y_test, y_pred)
report("Persistence", y_test, y_persist)
report("Linear",      y_test, y_linear)

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(vehicle_test["cycle_idx"], vehicle_test["soh_pct"], color="lightgray", label="Vehicle 14 actual SOH (full series)")
plt.plot(test_cycle_idx, y_test, "k",  lw=1.5, label="Vehicle 14 actual SOH (test region)")
plt.plot(test_cycle_idx, y_pred, "C0", lw=1.2, label="Transformer prediction")
# plt.plot(test_cycle_idx, y_linear,  "C2--", lw=1.0, label="Linear (Vehicle 14 warmup)")
# plt.plot(test_cycle_idx, y_persist, "C3:",  lw=1.0, label="Persistence")
plt.axvline(WINDOW - 1, color="red", ls="--", lw=1, label=f"prediction start (segment {WINDOW})")
plt.xlabel("Vehicle 14 segment index"); plt.ylabel("SOH %")
plt.title("Vehicle 14 forecast — model trained on Vehicles 1, 3, 8, 10, 15 only")
plt.legend()
plt.show()

## 9. Results dataframe (actual vs predicted SOH for Vehicle 14)

In [ ]:
test_rows = vehicle_test.loc[test_cycle_idx].reset_index(drop=True)
results_df = pd.DataFrame({
    "cycle_idx":     test_cycle_idx,
    "t_start":       test_rows["t_start"].values,
    "t_end":         test_rows["t_end"].values,
    "soh_actual":    y_test,
    "soh_predicted": y_pred,
})
results_df["abs_error"] = (results_df["soh_actual"] - results_df["soh_predicted"]).abs()
print(f"{len(results_df)} test rows  |  MAE = {results_df['abs_error'].mean():.3f}")
results_df.head()

## 10. Export to CSV

- `vehicle_14_multivehicle_actual_predicted.csv` — actual-vs-predicted table for Vehicle 14 test segments only.
- `multivehicle_all_actual_soh.csv` — actual smoothed SOH for **all six vehicles**, every segment (including training-vehicle segments and Vehicle 14 warm-up).

Uncomment the `to_csv` lines to write to disk.

In [ ]:
# results_df.to_csv("vehicle_14_multivehicle_actual_predicted.csv", index=False)
print(f"Prepared vehicle_14_multivehicle_actual_predicted.csv  ({len(results_df)} rows)")

all_actual_blocks = []
for vid, cyc in all_cycles.items():
    block = cyc[["vehicle_id", "cycle_idx", "t_start", "t_end", "soh_pct"]].copy()
    block = block.rename(columns={"soh_pct": "soh_actual_smoothed"})
    block["role"] = VEHICLE_CONFIG[vid]["role"]
    all_actual_blocks.append(block)
all_actual_df = pd.concat(all_actual_blocks, ignore_index=True)
# all_actual_df.to_csv("multivehicle_all_actual_soh.csv", index=False)
print(f"Prepared multivehicle_all_actual_soh.csv  ({len(all_actual_df)} rows across {len(all_cycles)} vehicles)")
all_actual_df.head()

## Notes & next steps

- **Per-vehicle SOH calibration:** each vehicle uses its own first `REF_HEAD` full-charge segments as the dQ/ΔV reference, so every SOH curve starts near 100%. To re-tune Vehicle 14 specifically, edit `FULL_DELTA_SOC` / `MIN_DELTA_V` / `REF_HEAD` and re-run from step 4 onward — training does not need to be repeated unless those changes alter the training-cell SOH labels.
- **No `vehicle_id` feature** means the model has to generalise. If you want to allow it, add a one-hot or scalar `vehicle_id` to `FEATURE_COLS`; Vehicle 14 will need a "novel" embedding strategy at inference.
- **Validation is shuffled across training vehicles.** This rewards cell-agnostic behaviour. For a stricter holdout, switch to the *last 10 % of each training vehicle* (chronological) instead of a random shuffle.
- **Window length 180 segments.** If any vehicle has fewer than ~180 segments after the 1000-segment cap, `make_windows` returns an empty array — reduce `WINDOW` or check the load step.
- **Baselines** (Persistence and Linear) are reported so you can see whether the Transformer's MAE on Vehicle 14 is genuinely better than naive forecasts derived from Vehicle 14 alone.